# ML Project / NASDAQ Prediction and Analysis

**Objective** : Predicting possible long/short entries on the NASDAQ index, using Machine Learning algorithms and other tools to carry this project.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import requests 
import io
import os

In [3]:
path_to_data = ("./data")

In [4]:
html = requests.get("https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies", headers={"User-Agent": "Mozilla/5.0"}).text
table = pd.read_html(io.StringIO(html))
ndx = next(t for t in table if "Ticker" in t.columns)
TICKERS = ndx["Ticker"].str.replace(".", "-", regex=False).tolist()

In [5]:
len(TICKERS)

101

In [6]:
test = yf.download("AAPL", start="2018-01-01", end="2025-12-31",
                     auto_adjust=True, progress=False)
print(test.columns)
print(test.tail())

MultiIndex([( 'Close', 'AAPL'),
            (  'High', 'AAPL'),
            (   'Low', 'AAPL'),
            (  'Open', 'AAPL'),
            ('Volume', 'AAPL')],
           names=['Price', 'Ticker'])
Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2025-12-23  271.620667  271.760301  268.828279  270.104803  29642000
2025-12-24  273.066711  274.682309  271.461097  271.600701  17910600
2025-12-26  272.657837  274.622490  272.119294  273.415784  21521800
2025-12-29  273.016876  273.615223  271.610700  271.949773  23715200
2025-12-30  272.338715  273.336000  271.540898  272.069458  22139600


In [7]:
raw = yf.download(TICKERS, start="2018-01-01", end="2025-12-31",
                  auto_adjust=True, group_by="ticker",
                  progress=False, threads=True)

$HONA: Data doesn't exist for startDate = 1514782800, endDate = 1767157200
$SPCX: Data doesn't exist for startDate = 1514782800, endDate = 1767157200

2 Failed downloads:
['HONA', 'SPCX']: Data doesn't exist for startDate = 1514782800, endDate = 1767157200


In [8]:
raw['SPCX'].head()

Price,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-08,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
coverage_rows = []
available = raw.columns.get_level_values(0)
for ticker in TICKERS:
    if ticker in available:
        d = raw[ticker].dropna(how="all")
        coverage_rows.append({"ticker": ticker, "days": len(d),
                      "first_date": d.index.min().date() if len(d) else None,
                      "last_date": d.index.max().date() if len(d) else None})
    else:
        coverage_rows.append({"ticker": ticker, "days": 0, "first_date": None, "last_date": None})

coverage = pd.DataFrame(coverage_rows)

print(coverage.sort_values("days").head(10))   
print("Tickers without data:", (coverage["days"] == 0).sum())

   ticker  days  first_date   last_date
87   SPCX     0        None        None
43   HONA     0        None        None
29   CRWV   191  2025-03-28  2025-12-30
84   SNDK   221  2025-02-13  2025-12-30
66   NBIS   299  2024-10-21  2025-12-30
15   ALAB   447  2024-03-20  2025-12-30
13    ARM   576  2023-09-14  2025-12-30
41   GEHC   762  2022-12-15  2025-12-30
27    CEG   991  2022-01-19  2025-12-30
12    APP  1184  2021-04-15  2025-12-30
Tickers without data: 2


In [10]:
print(coverage.sort_values("days").head(20))

   ticker  days  first_date   last_date
87   SPCX     0        None        None
43   HONA     0        None        None
29   CRWV   191  2025-03-28  2025-12-30
84   SNDK   221  2025-02-13  2025-12-30
66   NBIS   299  2024-10-21  2025-12-30
15   ALAB   447  2024-03-20  2025-12-30
13    ARM   576  2023-09-14  2025-12-30
41   GEHC   762  2022-12-15  2025-12-30
27    CEG   991  2022-01-19  2025-12-30
12    APP  1184  2021-04-15  2025-12-30
2    ABNB  1269  2020-12-10  2025-12-30
36   DASH  1270  2020-12-09  2025-12-30
81   RKLB  1280  2020-11-24  2025-12-30
73   PLTR  1319  2020-09-30  2025-12-30
33   DDOG  1579  2019-09-19  2025-12-30
31   CRWD  1648  2019-06-12  2025-12-30
62   MRNA  1775  2018-12-07  2025-12-30
77    PDD  1868  2018-07-26  2025-12-30
58   MCHP  2010  2018-01-02  2025-12-30
72   PCAR  2010  2018-01-02  2025-12-30


In [11]:
df = yf.download('^NDX')
df = pd.DataFrame(df)

[*********************100%***********************]  1 of 1 completed


# what to do 

basically our model will predict if the market is going to be up or down at cloe , so we have to do create a new column y : which will be what we try to predict (up or down : calculated by the diff from open to close). ALSO we need more than open/close/low/high/ paramets we need information about the stock so that we can try to predict

In [12]:
df

Price,Close,High,Low,Open,Volume
Ticker,^NDX,^NDX,^NDX,^NDX,^NDX
Date,,,,,
2026-09-10,29103.509766,29250.410156,29038.220703,29097.339844,7174170000
2026-09-11,29368.439453,29473.119141,29313.509766,29331.480469,6587900000
2026-09-14,29127.160156,29285.500000,28867.669922,28890.929688,7179780000
2026-09-15,28937.839844,29142.800781,28899.429688,29100.160156,7934200000
2026-09-16,28945.060547,29235.890625,28753.289062,29110.039062,9968640000
2026-09-17,29446.980469,29494.949219,29305.900391,29401.539062,9884890000
2026-09-18,29644.169922,29647.800781,29371.890625,29529.640625,13286960000
2026-09-21,30482.349609,30557.339844,29933.220703,29947.160156,8813260000


In [13]:
def win(open,close) -> str:
    if open >= close :
        return "up"
    return "down"


In [14]:
df.columns = df.columns.get_level_values(0)
df["up/down"] = df.apply(lambda row: win(row["Open"], row["Close"]), axis=1)

In [15]:
df

Price,Close,High,Low,Open,Volume,up/down
Date,,,,,,
2026-09-10,29103.509766,29250.410156,29038.220703,29097.339844,7174170000,down
2026-09-11,29368.439453,29473.119141,29313.509766,29331.480469,6587900000,down
2026-09-14,29127.160156,29285.500000,28867.669922,28890.929688,7179780000,down
2026-09-15,28937.839844,29142.800781,28899.429688,29100.160156,7934200000,up
2026-09-16,28945.060547,29235.890625,28753.289062,29110.039062,9968640000,up
2026-09-17,29446.980469,29494.949219,29305.900391,29401.539062,9884890000,down
2026-09-18,29644.169922,29647.800781,29371.890625,29529.640625,13286960000,down
2026-09-21,30482.349609,30557.339844,29933.220703,29947.160156,8813260000,down
2026-09-22,30732.400391,30770.630859,30496.429688,30496.429688,9334690000,down


In [16]:
df.to_feather("data/data.feather")